In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from Functions import (laydowncoordinates, compute_A, energy, propose_move, valid_fold)

np.random.seed(0)  # optional, for reproducibility across reruns

H = 1
P = 0
Gamma = [H,P,P,H]
A = compute_A(Gamma)
N = len(Gamma)

In [4]:
def run_sa_trial(Gamma, A, N, beta, propose_move, laydowncoordinates, valid_fold, energy, H):
    while True:
        d = np.random.randint(0, 4, size=N - 1)
        coordinates = laydowncoordinates(d)
        if valid_fold(coordinates):
            break

    E = energy(coordinates, Gamma, A, H)
    best_E = E

    for i in range(len(beta)):
        while True:
            d_new = propose_move(d)
            coords_new = laydowncoordinates(d_new)
            if valid_fold(coords_new):
                break
        E_new = energy(coords_new, Gamma, A, H)
        deltaE = E_new - E

        p_accept = 1.0 / (1.0 + np.exp(deltaE * beta[i]))
        if np.random.rand() < p_accept:
            d, coordinates, E = d_new, coords_new, E
            E = E_new
            if E < best_E:
                best_E = E

    return best_E, E

In [5]:
def schedules(num_sweeps, reinforce_betas=None, num_stairs=5):
    return {
        "Linear": np.linspace(0.1, 5, num_sweeps),
        "Geometric": 0.1 * (5 / 0.1) ** (np.arange(num_sweeps) / (num_sweeps - 1)),
        "Logarithmic": 0.1 + (5 - 0.1) * np.log1p(np.arange(num_sweeps)) / np.log1p(num_sweeps - 1),
        "Linear Staircase": np.repeat(np.linspace(0.1, 5, num_stairs), num_sweeps // num_stairs),
        "Geometric Staircase": np.repeat(0.1 * (5 / 0.1) ** (np.arange(num_stairs) / (num_stairs - 1)), num_sweeps // num_stairs),
        "Quenching": np.full(num_sweeps, 5.0),
        "REINFORCE": reinforce_betas,
    }

In [6]:
test_beta = np.linspace(0.1, 5, 30)
t0 = time.time()
best_E, final_E = run_sa_trial(Gamma, A, N, test_beta, propose_move,
                                laydowncoordinates, valid_fold, energy, H)
elapsed = time.time() - t0
print(f"one trial ({len(test_beta)} sweeps) took {elapsed:.4f}s -> best_E={best_E}, final_E={final_E}")

num_trials = 100
num_sweeps = 30
num_schedules = 6  # excluding REINFORCE for now

est_per_trial = elapsed          # already timed at the real num_sweeps, no scaling needed
est_total = est_per_trial * num_trials * num_schedules
print(f"estimated total time for full benchmark: {est_total/60:.2f} minutes")

one trial (30 sweeps) took 0.0036s -> best_E=-1.0, final_E=-1.0
estimated total time for full benchmark: 0.04 minutes


In [7]:
target_E = -1
num_trials = 100
num_sweeps = 30
PRINT_EVERY = 20 # print progress every N trials (adjust as you like)

sched_dict = schedules(num_sweeps, reinforce_betas=None)  # REINFORCE not ready yet
results = {name: {"best": [], "final": []} for name in sched_dict.keys()}

overall_start = time.time()

for name, beta in sched_dict.items():
    if beta is None:
        print(f"skipping {name} (no schedule provided)")
        continue

    print(f"\n=== running {name} ({num_trials} trials, {num_sweeps} sweeps each) ===")
    schedule_start = time.time()

    for trial in range(num_trials):
        best_E, final_E = run_sa_trial(Gamma, A, N, beta, propose_move,
                                        laydowncoordinates, valid_fold, energy, H)
        results[name]["best"].append(best_E)
        results[name]["final"].append(final_E)

        if (trial + 1) % PRINT_EVERY == 0 or (trial + 1) == num_trials:
            elapsed = time.time() - schedule_start
            rate = (trial + 1) / elapsed
            remaining = (num_trials - (trial + 1)) / rate
            running_best_mean = np.mean(results[name]["best"])
            print(f"  [{name}] trial {trial+1}/{num_trials} "
                  f"({elapsed:.1f}s elapsed, ~{remaining:.1f}s remaining) "
                  f"running mean best_E={running_best_mean:.3f}")

    print(f"=== {name} done in {time.time()-schedule_start:.1f}s ===")

print(f"\nTOTAL benchmark time: {(time.time()-overall_start)/60:.1f} minutes")


=== running Linear (100 trials, 30 sweeps each) ===
  [Linear] trial 20/100 (0.0s elapsed, ~0.1s remaining) running mean best_E=-1.000
  [Linear] trial 40/100 (0.1s elapsed, ~0.2s remaining) running mean best_E=-1.000
  [Linear] trial 60/100 (0.2s elapsed, ~0.1s remaining) running mean best_E=-0.983
  [Linear] trial 80/100 (0.3s elapsed, ~0.1s remaining) running mean best_E=-0.988
  [Linear] trial 100/100 (0.3s elapsed, ~0.0s remaining) running mean best_E=-0.990
=== Linear done in 0.3s ===

=== running Geometric (100 trials, 30 sweeps each) ===
  [Geometric] trial 20/100 (0.0s elapsed, ~0.2s remaining) running mean best_E=-0.950
  [Geometric] trial 40/100 (0.1s elapsed, ~0.2s remaining) running mean best_E=-0.975
  [Geometric] trial 60/100 (0.2s elapsed, ~0.1s remaining) running mean best_E=-0.983
  [Geometric] trial 80/100 (0.2s elapsed, ~0.0s remaining) running mean best_E=-0.988
  [Geometric] trial 100/100 (0.3s elapsed, ~0.0s remaining) running mean best_E=-0.990
=== Geometric do

In [8]:
rows = []
for name, d in results.items():
    if not d["best"]:
        continue
    best_arr, final_arr = np.array(d["best"]), np.array(d["final"])
    rows.append({
        "Schedule": name,
        "Success Probability": np.mean(best_arr == target_E),
        "Mean Best Energy": best_arr.mean(),
        "Std Best Energy": best_arr.std(),
        "Mean Final Energy": final_arr.mean(),
        "Std Final Energy": final_arr.std(),
    })

df = pd.DataFrame(rows)
print(df)

              Schedule  Success Probability  Mean Best Energy  \
0               Linear                 0.99             -0.99   
1            Geometric                 0.99             -0.99   
2          Logarithmic                 1.00             -1.00   
3     Linear Staircase                 1.00             -1.00   
4  Geometric Staircase                 1.00             -1.00   
5            Quenching                 1.00             -1.00   

   Std Best Energy  Mean Final Energy  Std Final Energy  
0         0.099499              -0.90          0.300000  
1         0.099499              -0.82          0.384187  
2         0.000000              -0.98          0.140000  
3         0.000000              -0.91          0.286182  
4         0.000000              -0.92          0.271293  
5         0.000000              -0.97          0.170587  


In [9]:
df.to_csv("sa_schedule_comparison_results.csv", index=False)
np.savez("sa_schedule_raw_results.npz",
         **{f"{name}_best": np.array(d["best"]) for name, d in results.items() if d["best"]},
         **{f"{name}_final": np.array(d["final"]) for name, d in results.items() if d["final"]})
print("saved.")

saved.


In [ ]:
target_E = -1
rows = []
for name, d in results.items():
    if not d["best"]:
        continue
    best_arr, final_arr = np.array(d["best"]), np.array(d["final"])
    rows.append({
        "Schedule": name,
        "Success Probability": np.mean(final_arr == target_E),
        "Mean Best Energy": best_arr.mean(),
        "Std Best Energy": best_arr.std(),
        "Mean Final Energy": final_arr.mean(),
        "Std Final Energy": final_arr.std(),
    })
df = pd.DataFrame(rows)
print(df)

              Schedule  Success Probability  Mean Best Energy  \
0               Linear                  0.0             -0.99   
1            Geometric                  0.0             -0.99   
2          Logarithmic                  0.0             -1.00   
3     Linear Staircase                  0.0             -1.00   
4  Geometric Staircase                  0.0             -1.00   
5            Quenching                  0.0             -1.00   

   Std Best Energy  Mean Final Energy  Std Final Energy  
0         0.099499              -0.90          0.300000  
1         0.099499              -0.82          0.384187  
2         0.000000              -0.98          0.140000  
3         0.000000              -0.91          0.286182  
4         0.000000              -0.92          0.271293  
5         0.000000              -0.97          0.170587  


In [11]:
import pickle

with open("all_results_backup.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "results": results,
    }, f)

print("saved: sa_benchmark_summary.csv, sa_benchmark_raw_trials.csv, all_results_backup.pkl")

saved: sa_benchmark_summary.csv, sa_benchmark_raw_trials.csv, all_results_backup.pkl
